# Run Match

In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100`% !important; }</style>"))
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

In [2]:
from timeutils import Timestat
from master import MasterParams, MasterDBs, MasterPaths, MasterMetas, MusicDBPermDir
from musicdb import PanDBIO
from gate import MusicDBGate
from pandas import DataFrame, Series, concat
import musicdb
from ioutils import HTMLIO, FileIO
from listUtils import getFlatList
import swifter
import dask.dataframe as dd
from match import MatchDBDataIO, AlbumReq, NameReq, MatchReq, MatchDB, CrossMatchDB, PanDBMatch
io = FileIO()

In [3]:
baseReqs = {"MetalArchives": 4,
            "RateYourMusic": 3,
            "Beatport": 35,
            "Spotify": 20,
            "Discogs": 3,  ## 12
            "Traxsource": 100000}
#baseDB    = "Beatport"
#baseDB    = "Discogs"
#baseDB    = "Spotify"
#baseDB    = "Traxsource"
#baseDB    = "MyMixTapez"
#baseDB    = "Genius"
#baseDB    = "MetalArchives"
baseDB    = "RateYourMusic"  # 3

minL = 1
maxL = 8

minA = 1
maxA = 30000000

#baseReq   = {baseDB: MatchReq(NameReq(min=minL, max=maxL), AlbumReq(min=baseReqs.get(baseDB), max=baseReqs.get(baseDB)+1))}
baseReq   = {baseDB: MatchReq(NameReq(min=minL, max=maxL), AlbumReq(min=minA, max=maxA))}
#baseReq   = {baseDB: AlbumReq(min=baseReqs.get(baseDB), max=baseReqs.get(baseDB)+1)}
#baseReq   = {baseDB: AlbumReq(min=10, max=baseReqs.get(baseDB,10000)+1, rnd=10000)}

compareDBs  = ["RateYourMusic", "LastFM", "Spotify", "Genius", "Discogs", "MusicBrainz", "Deezer", "MetalArchives"]
compareDBs  = ["RateYourMusic", "Spotify", "Genius", "Discogs", "MusicBrainz", "LastFM", "Deezer"]
compareDBs  = ["RateYourMusic", "Spotify", "Genius", "Discogs", "MusicBrainz", "Beatport"] # "LastFM", "Deezer"]
compareDBs  = ["RateYourMusic", "Spotify", "Discogs", "MusicBrainz", "Beatport", "Traxsource", "Genius", "MyMixTapez", "MetalArchives"] # "LastFM", "Deezer"]
#compareDBs  = ["RateYourMusic", "Traxsource", "Spotify", "Beatport"]
compareReqs = {compareDB: MatchReq(NameReq(min=minL-5, max=maxL+5), AlbumReq(min=3)) for compareDB in compareDBs if compareDB not in [baseDB]}
compareDBs  = list(compareReqs.keys())

matchReqs  = {**baseReq, **compareReqs}
mediaTypes = ["Album", "SingleEP"]
mediaTypes = ["{0}Media".format(mediaType) for mediaType in mediaTypes]
mediaTypes = list(MasterMetas().getMedias().values())
reqs       = {"Media": mediaTypes, "Reqs": matchReqs, "Mask": baseDB, "NPart": 3, "Match": {"Artist": 0.85, "Medium": 2, "Tight": 1}}
print("Primary Run Params:")
print("  ==> DBs:   [{0}] x {1}]".format(baseDB,list(compareReqs.keys())))
print("  ==> Media: {0}".format(mediaTypes))
print("  ==> Match: {0}".format(reqs["Match"]))

crossreqs  = {"Media": mediaTypes, "Reqs": {baseDB: MatchReq(AlbumReq(min=2))}, "Mask": baseDB, "NPart": 3, "Match": {"Artist": 0.85, "Medium": 2, "Tight": 1}}
print("Cross Match Run Params:")
print("  ==> DBs:   {0} x [{1}]".format(list(compareReqs.keys()), baseDB))
print("  ==> Media: {0}".format(mediaTypes))
print("  ==> Match: {0}".format(crossreqs["Match"]))

Primary Run Params:
  ==> DBs:   [RateYourMusic] x ['Spotify', 'Discogs', 'MusicBrainz', 'Beatport', 'Traxsource', 'Genius', 'MyMixTapez', 'MetalArchives']]
  ==> Media: ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Match: {'Artist': 0.85, 'Medium': 2, 'Tight': 1}
Cross Match Run Params:
  ==> DBs:   ['Spotify', 'Discogs', 'MusicBrainz', 'Beatport', 'Traxsource', 'Genius', 'MyMixTapez', 'MetalArchives'] x [RateYourMusic]
  ==> Media: ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Match: {'Artist': 0.85, 'Medium': 2, 'Tight': 1}


In [ ]:
baseIO = MatchDBDataIO(db=baseDB, mediaTypes=reqs["Media"], mask=reqs["Mask"], verbose=True, base=True)
baseIO.loadNames()
baseIO.setAvailableNames(reqs["Reqs"][baseDB])
del baseIO

# Match & Cross Match

In [ ]:
mdb = MatchDB(baseDB=baseDB, compareDBs=compareDBs, reqs=reqs)
mdb.match()
mdb.save()
del mdb

In [ ]:
mdbpd = MusicDBPermDir()
mres  = io.get(mdbpd.getMatchPermPath().join("primaryMatch.p"))
cmdb  = CrossMatchDB(baseDB, mres, crossreqs, verbose=True)
cmdb.match()
cmdb.save()

del cmdb

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=5)
pdbm.master()
pdbm.merge(allownew=True, verbose=False)

# Extra Match

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=4, maxQual=5)

In [ ]:
pdbm.include("""
6937a63928   | 637067                   Genius         1069488                                 4    JO KWON                                           JO KWON                                            | 6937a63928
""")
pdbm.master()
pdbm.merge()

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=2, maxQual=4)

In [ ]:
pdbm.include("""
a8f7b31ccc   | 36444                    Genius         381188                                  3    TOM CLAY                                          TOM CLAY                                           | a8f7b31ccc
""")
pdbm.master()
pdbm.merge()

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=1, maxQual=2)

In [ ]:
pdbm.include("""
0ff972c6a6   | 1335340                  Spotify        1yewRvlKGWmNRHOSGgiRRo                  1    HELLO GAYOUNG                                     HELLO GA-YOUNG                                     | 0ff972c6a6
957c2db76c   | 174489                   Spotify        5W9v4joSRnv2hSQNYkxExx                  1    WILLY BERKING                                     WILLY BERKING                                      | 957c2db76c
93143646f5   | 175975                   Genius         1824571                                 1    EDNA HICKS                                        EDNA HICKS                                         | 93143646f5
7233ba29a2   | 688342                   Discogs        1117347                                 1    THE MARK IV                                       THE MARK IV                                        | 7233ba29a2
913454b605   | 82649                    Spotify        6b3OgUgW0vGjuE1nf9Ap0I                  1    THE BELIEVERS                                     THE BELIEVERS                                      | 913454b605
""")

pdbm.master()
pdbm.merge()

In [ ]:
del pdbm

In [ ]:
# Fix This:
#afc404588f   | 183166                   Discogs        113700                                  2    STEVE MASTERSON                                   STEVE MAESTRO                                      | afc404588f
#a6da8c9ee9   | 25875                    Discogs        678236                                  4    ALTAR OF FLESH                                    ALTAR OF FLIES                                     | a6da8c9ee9

# New Matching Code

In [ ]:
class CrossMatchDB(MatchDBBase):
    def __init__(self, compareDB, mres: DataFrame, reqs: dict, **kwargs):
        super().__init__(reqs, **kwargs)
        
        self.compareIO = MatchDBDataIO(db=compareDB, mediaTypes=self.params.mediaTypes, mask=self.params.mask, verbose=False, base=False)
        self.validDB(self.compareIO.db)
        
        self.mres = mres
        self.cmres = CrossMatchResults()
        self.save = self.cmres.save
        

    def match(self, **kwargs):
        verbose = kwargs.get('verbose', self.verbose)
        baseDBs = list(self.mres["DB"].unique())
        tsMatch = Timestat("Cross Matching [{0}] Against {1}".format(self.compareIO.db, baseDBs), ind=0)
        
        compareIO = self.compareIO        
        compareIO.loadNames()
        compareIO.setAvailableNames(self.getDBReq(compareIO.db))
                
        index = self.mres.apply(lambda row: (row["BaseID"],row["DB"],row["CompareID"]), axis=1)
        baseNames = self.mres['Match'].apply(lambda x: x["Info"]["Name"])
        baseMediaData = self.mres["Match"].apply(lambda x: x["Media"])
        baseNames.index = index
        baseNames.name = "Name"
        baseMediaData.index = index
        baseMediaData.name = "Media"
        self.cmres.setBaseNames(baseNames)

        ########################################################################################################################################
        ## 1) Match Artist Names
        ########################################################################################################################################
        if verbose: ts = Timestat("String Matching {0} {1} x {2} [{3}] Artist Names".format(len(baseNames), baseDBs, compareIO.getNumNames(), compareIO.db), ind=2)
        artistMatchResults = poolMatchNames(baseNames=baseNames, compNames=compareIO.getAvailableNames(), nCores=self.getPart(), progress=True)
        artistNameMatches  = self.selectArtistsForMediaMatch(artistMatchResults)
        mediaData          = self.prepareMediaData(artistNameMatches, baseMediaData, compareIO)
        del artistMatchResults
        del artistNameMatches
        if verbose: ts.stop()
            
            
        ########################################################################################################################################
        ## 2) Match Artist Albums Names
        ########################################################################################################################################
        if verbose: ts = Timestat("String Matching {0} {1} Album Names".format(mediaData.shape[0], baseDBs), ind=2)
        albumMatchResults = self.matchMediaDataPool(mediaData)
        self.cmres.addResult(compareIO.db, compareIO, albumMatchResults)
        del albumMatchResults
        del mediaData
        if verbose: ts.stop()
                
        del compareIO

    
    ################################################################################################################################################
    ## Prepare Media Data For Match
    ################################################################################################################################################
    def prepareMediaData(self, artistNameMatches: Series, baseMediaData: Series, compareIO: MatchDBDataIO) -> 'DataFrame':
        nameMatchValues = {}
        for baseid,compareValues in artistNameMatches.iteritems():
            for compareid,value in compareValues.items():
                key   = (baseid,compareid)
                nameMatchValues[key] = value
        compids = [compid for compid,_ in Series(nameMatchValues).groupby(level=1)]        
        compareIO.loadMedia(ids=compids)
        compareMediaData = compareIO.getAvailableMedia()        

        mediaData = {}
        for key in nameMatchValues.keys():
            baseid,compid = key
            mediaData[key] = {"Base": Series(baseMediaData[baseid]), "Compare": Series(compareMediaData[compid])}
        mediaData = Series(mediaData)
        return mediaData

In [ ]:
class PrimaryMatchDB(MatchDBBase):
    def __init__(self, baseDB: str, compareDBs: list, reqs: dict, **kwargs):
        super().__init__(reqs, **kwargs)
        
        self.baseIO = MatchDBDataIO(db=baseDB, mediaTypes=self.params.mediaTypes, mask=self.params.mask, verbose=False, base=True)
        self.validDB(self.baseIO.db)
        
        self.compareIOs = {}
        for compareDB in compareDBs:
            compareIO = MatchDBDataIO(db=compareDB, mediaTypes=self.params.mediaTypes, mask=self.params.mask, verbose=False, base=False)
            self.validDB(compareIO.db)
            self.compareIOs[compareDB] = compareIO
        self.compareDBs = compareDBs
        
        self.diagnostics = {}
        self.mres = PrimaryMatchResults(self.baseIO)
        self.save = self.mres.save
        

    def match(self, **kwargs):
        verbose = kwargs.get('verbose', self.verbose)                
        tsMatch = Timestat(f"Matching [{self.baseIO.db}] Against {self.compareDBs}", ind=0)
        
        baseIO = self.baseIO
        printIntro(baseIO.db, delimiter='-')
        if verbose: ts = Timestat(f"Loading {baseIO.db} Artist Names", ind=2)
        baseIO.loadNames()
        baseIO.setAvailableNames(self.getDBReq(baseIO.db))
        if verbose: ts.stop()
        
        ###################################################################################################################################################
        ## Loop Through Compare DBs
        ###################################################################################################################################################
        for compareDB in self.compareIOs.keys():
            compareIO = self.compareIOs[compareDB]
            printIntro(compareDB, delimiter='-')
                        
            if verbose: ts = Timestat(f"Loading {compareDB} Artist Names", ind=2)
            compareIO.loadNames()
            compareIO.setAvailableNames(self.getDBReq(compareIO.db))
            if verbose: ts.stop()
        
            ########################################################################################################################################
            ## 1) Match Artist Names
            ########################################################################################################################################
            if verbose: ts = Timestat(f"String Matching {baseIO.getNumNames()} [{baseIO.db}] x {compareIO.getNumNames()} [{compareIO.db}] Artist Names", ind=2)
            artistMatchResults = poolMatchNames(baseNames=baseIO.getAvailableNames(), compNames=compareIO.getAvailableNames(), nCores=self.getPart(), progress=True, cutoff=self.getMatchNameReq())
            if verbose: ts.stop()
                
            artistNameMatches  = self.selectArtistsForMediaMatch(artistMatchResults)
            if artistNameMatches.shape[0] == 0:
                del artistMatchResults
                self.compareIOs[compareDB] = None
                continue                
            mediaData          = self.prepareMediaData(artistNameMatches, baseIO, compareIO)
            del artistMatchResults
            del artistNameMatches
            
            
            ########################################################################################################################################
            ## 2) Match Artist Albums Names
            ########################################################################################################################################
            if verbose: ts = Timestat(f"String Matching {mediaData.shape[0]} [{baseIO.db}] Album Names", ind=2)
            albumMatchResults = self.matchMediaDataPool(mediaData)
            self.mres.addResult(compareDB, compareIO, albumMatchResults)
            del albumMatchResults
            del mediaData
            if verbose: ts.stop()
                
            self.compareIOs[compareDB] = None
        
        tsMatch.stop()
    
    
    ################################################################################################################################################
    ## Prepare Media Data For Match
    ################################################################################################################################################
    def prepareMediaData(self, artistNameMatches: Series, baseIO: MatchDBDataIO, compareIO: MatchDBDataIO) -> 'DataFrame':
        nameMatchValues = {}
        for baseid,compareValues in artistNameMatches.iteritems():
            for compareid,value in compareValues.items():
                key   = (baseid,compareid)
                nameMatchValues[key] = value
        baseids = [baseid for baseid,_ in Series(nameMatchValues).groupby(level=0)]
        compids = [compid for compid,_ in Series(nameMatchValues).groupby(level=1)]

        if self.verbose: ts = Timestat("Loading {0} Media Data".format(baseIO.db), ind=4)
        baseIO.loadMedia()
        baseMediaData = baseIO.getAvailableMedia()
        if self.verbose: ts.stop()
            
        if self.verbose: ts = Timestat("Loading {0} Media Data".format(compareIO.db), ind=4)
        compareIO.loadMedia(ids=compids)
        compareMediaData = compareIO.getAvailableMedia()
        if self.verbose: ts.stop()
        
        mediaData = {}
        for key in nameMatchValues.keys():
            baseid,compid = key
            mediaData[key] = {"Base": Series(baseMediaData[baseid]), "Compare": Series(compareMediaData[compid])}
        mediaData = Series(mediaData)
        return mediaData

# New Single Matching Code

In [ ]:
names = smdb.baseIO.getAvailableNames()

In [ ]:
smdb = SingleMatchDB(baseDB="RateYourMusic", compareDB="Spotify", reqs=reqs)
smdb.match()
smdb.save()
del smdb


mdbpd = MusicDBPermDir()
mres  = io.get(mdbpd.getMatchPermPath().join("primaryMatch.p"))
scmdb = SingleCrossMatchDB(baseDB, mres, crossreqs, verbose=True)
scmdb.match()
scmdb.save()
del scmdb


In [ ]:
pdbio = PanDBIO()
mmeDF = pdbio.getData()

In [ ]:
mmeDF[mmeDF["RateYourMusic"] == '106836']

In [ ]:
mmeDF[mmeDF["Spotify"] == '3lk3F4u5qqxq8zFjwNj5U1']

In [4]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=5)
pdbm.masterSingle()
#pdbm.merge(allownew=False, verbose=False)


*******************************************************************************************************************************************************************************
**                                                                             PanDBMatch()                                                                                  **
*******************************************************************************************************************************************************************************
457cf936bc   | 106836                   Spotify        3lk3F4u5qqxq8zFjwNj5U1                  8    MARIA FARANTOURI                                  MARIA FARANTOURI                                   | 457cf936bc
ca9d8a40f8   | 1128593                  Spotify        4GLREhkqd6WaKIQjv3nIXb                  8    OYASUMI HOLOGRAM                                  OYASUMI HOLOGRAM                                   | ca9d8a40f8
f2f45ce69e   | 1298447                  Spo

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=4, maxQual=5)

In [ ]:
pdbm.include("""
1d2402a17a   | 1061693                  Spotify        49D8h67pxvvUNGOLKEGjkx                  4    OWAIN ARWEL HUGHES                                OWAIN ARWEL HUGHES                                 | 1d2402a17a
a86b2ef789   | 121809                   Spotify        6NSIW1uEq8JZmxEkHMF17c                  4    ANNA TOMOWA-SINTOW                                ANNA TOMOWA-SINTOW                                 | a86b2ef789
877d262f5e   | 142182                   Spotify        5DwQvVHPVspRvStEAN722N                  4    TAKÁCS QUARTET                                   TAKÁCS QUARTET                                    | 877d262f5e
a2f65f8447   | 412578                   Spotify        50skve7Y0al39yGqLuCMNu                  4    MAURICE ABRAVANEL                                 MAURICE ABRAVANEL                                  | a2f65f8447
""")
pdbm.masterSingle()
pdbm.merge(allownew=False, verbose=False)

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=2, maxQual=4)

In [ ]:
pdbm.include("""
9554709d72   | 405351                   Spotify        2mHCS8PPaV7cAmT3ew8qHY                  2    SAULIUS SONDECKIS                                 SAULIUS SONDECKIS                                  | 9554709d72
""")
pdbm.masterSingle()
pdbm.merge(allownew=False, verbose=False)

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=1, maxQual=2)

In [ ]:
pdbm.include("""
3178f6847b   | 337551                   Spotify        7N0fh2csz0eFkrE01LF1m3                  1    STRATOS PAGIOUMTZIS                               STRATOS PAGIOUMTZIS                                | 3178f6847b
842d333cee   | 375588                   Spotify        2LqWWIvCBaetjLStxk1XK6                  1    VAN AND SCHENCK                                   VAN & SCHENCK                                      | 842d333cee
60cc9bc61a   | 77193                    Spotify        6VeTIJi6Dlx8ywPfIwqALY                  1    ALBERT NICHOLAS                                   ALBERT NICHOLAS                                    | 60cc9bc61a
""")
pdbm.masterSingle()
pdbm.merge(allownew=False, verbose=False)